# Unit 2 (CO-2): ReAct Reasoning Agent with Dynamic Replanning

**Goal:** Plan a 2-night Pune → Goa student trip within a Rs 9,000 budget.

**Sub-tasks produced by the agent:**
1. Find transport Pune → Goa
2. Check the weather in Goa
3. Find a stay for 2 nights
4. Compute total cost and verify it against the budget
5. Write the final itinerary

## Code
The agent loops Thought → Action → Observation. Tools are simulated functions. The reasoning step is a rule-based policy so the run is repeatable; `think()` is where an LLM call could be swapped in.

In [5]:
"""ReAct (Reason + Act) agent with dynamic replanning.

Goal: plan a 2-night Pune -> Goa student trip under a Rs 9,000 budget.
The agent loops: Thought -> Action -> Observation, and rewrites its plan
when an observation contradicts an assumption.

The 'reasoning' step is a rule-based policy so the program runs offline
and the trace is reproducible. To use a real LLM, replace Agent.think()
with a model call that returns (thought, action, args).
"""

BUDGET = 9000
NIGHTS = 2

# ---------------- Tools (simulated environment) ----------------
def search_transport(mode):
    data = {
        "train": {"ok": False, "msg": "Train 12780 Pune->Goa: SOLD OUT for the travel date (waitlist 140)."},
        "bus":   {"ok": True, "price_each_way": 1400, "msg": "Sleeper bus Pune->Goa available, Rs 1400 each way."},
        "flight": {"ok": True, "price_each_way": 5200, "msg": "Flight available, Rs 5200 each way."},
    }
    return data[mode]

def check_weather(city):
    return {"ok": True, "msg": f"{city}: partly cloudy, 30C, low rain chance on both days."}

def find_stay(kind):
    data = {
        "beach_resort": {"ok": True, "price_per_night": 3500, "msg": "Beachfront resort in Calangute, Rs 3500/night."},
        "hostel": {"ok": True, "price_per_night": 1200, "msg": "Backpacker hostel in Baga, Rs 1200/night, 4.3 rating."},
    }
    return data[kind]

def compute_total(transport_one_way, stay_per_night):
    total = transport_one_way * 2 + stay_per_night * NIGHTS + 1000  # Rs 1000 food/local
    return {"ok": total <= BUDGET, "total": total,
            "msg": f"Total = 2x{transport_one_way} + {NIGHTS}x{stay_per_night} + 1000 food = Rs {total} "
                   f"({'within' if total <= BUDGET else 'OVER'} budget Rs {BUDGET})."}

TOOLS = {"search_transport": search_transport, "check_weather": check_weather,
         "find_stay": find_stay, "compute_total": compute_total}

# ---------------- Agent ----------------
class Agent:
    def __init__(self, goal):
        self.goal = goal
        self.plan = []          # list of [subtask, status]
        self.mem = {}           # facts learned from observations
        self.tried_transport = []
        self.tried_stay = []
        self.step = 0
        self.replans = 0

    def make_plan(self):
        self.plan = [[s, "pending"] for s in [
            "Find transport Pune -> Goa",
            "Check weather in Goa",
            "Find a stay for 2 nights",
            "Compute total cost and verify budget",
            "Write final itinerary",
        ]]
        print("PLAN (initial):")
        for i, (s, _) in enumerate(self.plan, 1):
            print(f"  {i}. {s}")
        print()

    def current(self):
        for s in self.plan:
            if s[1] == "pending":
                return s
        return None

    def think(self, last_obs):
        """Decide next (thought, action, args). Replans on failure."""
        task = self.current()[0]
        if task.startswith("Find transport"):
            mode = "train" if not self.tried_transport else ("bus" if "bus" not in self.tried_transport else "flight")
            if not self.tried_transport:
                th = "Assumption: the cheapest, most comfortable option is the train. Try it first."
            else:
                th = (f"The {self.tried_transport[-1]} option failed. Replan: switch transport to {mode}.")
            self.tried_transport.append(mode)
            return th, "search_transport", {"mode": mode}
        if task.startswith("Check weather"):
            return "Weather affects the plan (beach trip), so check it before booking a stay.", "check_weather", {"city": "Goa"}
        if task.startswith("Find a stay"):
            kind = "beach_resort" if not self.tried_stay else "hostel"
            if not self.tried_stay:
                th = "Assumption: a beachfront resort fits the budget. Try it first."
            else:
                th = "The resort made the trip exceed the budget. Replan: switch to a cheaper hostel."
            self.tried_stay.append(kind)
            return th, "find_stay", {"kind": kind}
        if task.startswith("Compute total"):
            return ("I have transport and stay prices. Verify the total against the budget.",
                    "compute_total",
                    {"transport_one_way": self.mem["transport"], "stay_per_night": self.mem["stay"]})
        return "All facts gathered and budget verified. Produce the final answer.", "finish", {}

    def observe_and_update(self, action, obs):
        task = self.current()
        if action == "search_transport":
            if obs["ok"]:
                self.mem["transport"] = obs["price_each_way"]
                self.mem["mode"] = self.tried_transport[-1]
                task[1] = "done"
            else:
                self.replans += 1
                print(f"  [REPLAN #{self.replans}] Assumption 'train is available' was wrong. "
                      f"Keeping sub-task, changing transport mode.")
        elif action == "check_weather":
            task[1] = "done"
        elif action == "find_stay":
            self.mem["stay"] = obs["price_per_night"]
            self.mem["stay_kind"] = self.tried_stay[-1]
            task[1] = "done"
        elif action == "compute_total":
            if obs["ok"]:
                self.mem["total"] = obs["total"]
                task[1] = "done"
            else:
                self.replans += 1
                # insert a fix: redo the stay search, then re-verify
                idx = self.plan.index(task)
                self.plan[idx - 1][1] = "pending"
                print(f"  [REPLAN #{self.replans}] Budget check failed (Rs {obs['total']} > Rs {BUDGET}). "
                      f"Re-opening the 'stay' sub-task and re-verifying afterwards.")

    def run(self):
        print(f"GOAL: {self.goal}\n")
        self.make_plan()
        obs = None
        while self.step < 15:
            self.step += 1
            th, action, args = self.think(obs)
            print(f"Step {self.step}")
            print(f"Thought: {th}")
            print(f"Action: {action}({', '.join(f'{k}={v!r}' for k, v in args.items())})")
            if action == "finish":
                m = self.mem
                print("Observation: (none, finishing)\n")
                print("FINAL ANSWER:")
                print(f"  Travel by {m['mode']} (Rs {m['transport']} each way), stay at "
                      f"{m['stay_kind'].replace('_', ' ')} (Rs {m['stay']}/night x {NIGHTS}), "
                      f"food/local Rs 1000. Total Rs {m['total']} <= budget Rs {BUDGET}.")
                print(f"  Replans during run: {self.replans}")
                return
            obs = TOOLS[action](**args)
            print(f"Observation: {obs['msg']}")
            self.observe_and_update(action, obs)
            print()
        print("Stopped: step limit reached.")




## Full run trace (Thought / Action / Observation)

In [6]:
Agent("Plan a 2-night Pune -> Goa student trip within a Rs 9,000 budget.").run()

GOAL: Plan a 2-night Pune -> Goa student trip within a Rs 9,000 budget.

PLAN (initial):
  1. Find transport Pune -> Goa
  2. Check weather in Goa
  3. Find a stay for 2 nights
  4. Compute total cost and verify budget
  5. Write final itinerary

Step 1
Thought: Assumption: the cheapest, most comfortable option is the train. Try it first.
Action: search_transport(mode='train')
Observation: Train 12780 Pune->Goa: SOLD OUT for the travel date (waitlist 140).
  [REPLAN #1] Assumption 'train is available' was wrong. Keeping sub-task, changing transport mode.

Step 2
Thought: The train option failed. Replan: switch transport to bus.
Action: search_transport(mode='bus')
Observation: Sleeper bus Pune->Goa available, Rs 1400 each way.

Step 3
Thought: Weather affects the plan (beach trip), so check it before booking a stay.
Action: check_weather(city='Goa')
Observation: Goa: partly cloudy, 30C, low rain chance on both days.

Step 4
Thought: Assumption: a beachfront resort fits the budget. Try 